# Долгосрочная память аптечного AI-ассистента · этап 3: оценка по слоям

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/serge3emskov/pharma-ltm-assistant/blob/main/notebooks/03_evaluation.ipynb)

**Golden set** — отложенная выборка: по ней промпты не настраивались (для настройки служил dev-набор этапа 2, тест проверяет, что почти-дублей нет). 97 сценариев: 85 одиночных сессий и 12 многосессионных с паузами до 240 дней, 20 проб контекста, 40 запросов.

Как в статье, проверяется вся цепочка, а не только ответ:

| Слой | Что измеряется |
|---|---|
| Классификатор разделов | P/R/F1 по разделам; отдельно — сколько добавила и спасла страховка кодом |
| Extractor | события (раздел, субъект, ключ после нормализации), точность полей, статусы разделов, цитата evidence есть в диалоге |
| Память | итоговые факты и статусы разделов после state machine, атрибуты, сценарии без ошибок |
| Retrieval | нужное в контексте, лишнего нет, верные «уточнить»: с эталонным типом запроса и с LLM-классификатором |
| E2E | факт из прошлой сессии учтён в ответе (эвристики по тексту ответа) |
| Классификатор запроса | точное совпадение типов против baseline на правилах |

Каждая ошибка памяти приписывается этапу: классификатор, пропуск extractor-а, статус, субъект, нормализация названия, валидация, каскад из прошлой сессии.

**Перед запуском:** Среда выполнения → Сменить среду выполнения → **T4 GPU**. Ответы модели генерируются пакетами и кэшируются на Google Drive: если сеанс оборвётся, перезапуск продолжит с того же места. Ориентировочно: Qwen3-4B — 10–15 минут, Qwen3-8B — 20–30 минут.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/serge3emskov/pharma-ltm-assistant.git"

if IN_COLAB:
    if os.path.exists("pharma-ltm-assistant"):
        subprocess.run(["git", "-C", "pharma-ltm-assistant", "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    ROOT = os.path.abspath("pharma-ltm-assistant")
    for req in ("requirements.txt", "requirements-llm.txt"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{ROOT}/{req}"], check=True)
else:
    ROOT = os.path.abspath("..")

SRC = os.path.join(ROOT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
# После git pull Python не перечитывает уже импортированный пакет — выгружаем старую версию
for name in [m for m in sys.modules if m == "pharmaltm" or m.startswith("pharmaltm.")]:
    del sys.modules[name]
import pharmaltm

commit = subprocess.run(["git", "-C", ROOT, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"pharmaltm {pharmaltm.__version__} · commit {commit or 'нет git'}")

## 1. Настройки

* `MODELS` — какие модели сравнивать. Вторую можно заменить, например на `Qwen/Qwen2.5-7B-Instruct`.
* `USE_DRIVE` — кэш ответов и отчёты на Google Drive (переживают обрыв сеанса).
* `BATCH_SIZE` — размер пакета генерации; при нехватке памяти уменьшается автоматически.
* `RUN_ANSWERS` — генерировать ответы ассистента для E2E-проверок.
* `FORCE` — пересчитать отчёт, даже если он уже сохранён.

In [ ]:
import gc, json, time
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from pharmaltm import CachedLLM, QwenTransformersLLM
from pharmaltm.evaluation import (compare_reports, gold_backend, load_golden, run_golden, summarize,
                                  to_report, warm_cache)
from pharmaltm.prompts import compact_dialog, extractor_messages
from pharmaltm import Section

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 240)

BACKEND = os.environ.get("PHARMALTM_BACKEND", "qwen")   # "gold" — проверка ноутбука без GPU
MODELS = ["Qwen/Qwen3-4B-Instruct-2507", "Qwen/Qwen3-8B"] if BACKEND == "qwen" else ["gold"]
USE_DRIVE = IN_COLAB
BATCH_SIZE = 8
RUN_ANSWERS = True
FORCE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/pharma-ltm")
else:
    WORK = Path(ROOT) / "runs"
CACHE_DIR, REPORT_DIR = WORK / "cache", WORK / "reports"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print("кэш:", CACHE_DIR, "| отчёты:", REPORT_DIR)

GOLDEN = load_golden()
stats = GOLDEN.stats()
print({k: v for k, v in stats.items() if k not in ("categories", "tags")})
display(pd.DataFrame([stats["categories"]], index=["сценариев"]))

## 2. Проверка разметки эталонным бэкендом

Эталонные ответы проходят через весь пайплайн. Все слои должны дать 100%: иначе ошибка в разметке или в коде, и сравнивать модели рано.

In [ ]:
gold_summary = summarize(run_golden(gold_backend(GOLDEN), GOLDEN, run_answers=False))
checks = {
    "классификатор": gold_summary["classifier"]["f1"],
    "extractor": gold_summary["extractor"]["events"]["f1"],
    "память": gold_summary["memory"]["facts"]["f1"],
    "статусы разделов": gold_summary["memory"]["section_states"]["f1"],
    "retrieval": gold_summary["retrieval"]["gold_query_types"]["probes_ok"] / gold_summary["retrieval"]["gold_query_types"]["probes"],
}
print(checks)
assert all(v == 1.0 for v in checks.values()), "разметка golden set расходится с кодом"
print("разметка согласована")

## 3. Прогон моделей

Для каждой модели: загрузка → контроль пакетной генерации → прогрев кэша пакетами (классификация, extraction, повторы) → последовательный прогон пайплайна из кэша → ответы для E2E → отчёт. Готовые отчёты не пересчитываются, если `FORCE = False`.

In [ ]:
def slug(model_id: str) -> str:
    return model_id.replace("/", "__")


def batch_sanity(base, n: int = 3) -> None:
    """Пакетная и одиночная генерация на одних промптах. Небольшие расхождения возможны из-за padding в fp16."""
    sessions = [s for _, _, s in GOLDEN.sessions()][:n]
    msgs = [extractor_messages(Section(s["gold_sections"][0]), compact_dialog(s["turns"])) for s in sessions if s["gold_sections"]]
    single = [base.chat(m, max_new_tokens=256) for m in msgs]
    batch = base.chat_batch(msgs, max_new_tokens=256, batch_size=len(msgs))
    same = sum(a == b for a, b in zip(single, batch))
    print(f"контроль пакетной генерации: совпало {same}/{len(msgs)}")
    for a, b in zip(single, batch):
        if a != b:
            print("  одиночно:", a[:200], "\n  пакетом: ", b[:200])


def evaluate_model(model_id: str) -> dict:
    path = REPORT_DIR / f"stage3_{slug(model_id)}.json"
    if path.exists() and not FORCE:
        print(f"{model_id}: отчёт уже есть — {path.name}")
        return json.loads(path.read_text(encoding="utf-8"))
    t0 = time.time()
    if BACKEND == "qwen":
        import torch
        print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет — включите T4")
        base = QwenTransformersLLM(model_id, load_in_4bit=True)
        print(f"{model_id} загружена за {time.time() - t0:.0f} с")
        batch_sanity(base)
    else:
        base = gold_backend(GOLDEN)
    llm = CachedLLM(base, str(CACHE_DIR / f"{slug(model_id)}.jsonl"))
    warm = warm_cache(llm, GOLDEN, batch_size=BATCH_SIZE)
    run = run_golden(llm, GOLDEN, run_answers=RUN_ANSWERS, batch_size=BATCH_SIZE, log=print)
    report = to_report(run, GOLDEN)
    report.update({"model": model_id, "commit": commit, "warm_timings": warm,
                   "total_minutes": round((time.time() - t0) / 60, 1)})
    path.write_text(json.dumps(report, ensure_ascii=False, indent=1, default=list), encoding="utf-8")
    print(f"{model_id}: готово за {report['total_minutes']} мин → {path.name}")
    if hasattr(base, "unload"):
        base.unload()
    del base, llm
    gc.collect()
    return report


reports = {}
for model_id in MODELS:
    reports[model_id] = evaluate_model(model_id)

## 4. Сравнение моделей

In [ ]:
comparison = pd.DataFrame(compare_reports(list(reports.values()))).set_index("метрика")
comparison

## 5. Ошибки памяти по этапам пайплайна

In [ ]:
pd.DataFrame({m: r["summary"]["errors_by_stage"] for m, r in reports.items()}).fillna(0).astype(int)

In [ ]:
for m, r in reports.items():
    display(Markdown(f"**{m}** — срезы с ошибками (по возрастанию доли сценариев без ошибок)"))
    display(pd.DataFrame(r["tags"]).query("`доля` < 1").head(15))

## 6. Слои подробно

In [ ]:
for m, r in reports.items():
    s = r["summary"]
    display(Markdown(f"### {m}"))
    display(Markdown("**Классификатор разделов по разделам**"))
    display(pd.DataFrame(s["classifier"]["per_section"]).T)
    display(Markdown("**Extractor: точность полей**"))
    display(pd.DataFrame(s["extractor"]["fields"]).T)
    guards = {k: s["extractor"][k] for k in ("json_failures", "validation_repairs", "validation_repaired",
                                             "rejected_validation", "rejected_reference", "off_section_dropped",
                                             "extra_text_attributes")}
    guards.update({"safety_net_added": s["classifier"]["safety_net_added"],
                   "safety_net_recovered": s["classifier"]["safety_net_recovered"]})
    display(Markdown("**Что поймал код**"))
    display(pd.DataFrame([guards]))
    display(Markdown("**Классификатор запроса по типам**"))
    display(pd.DataFrame(r["query_types"]).set_index("тип"))

## 7. Примеры ошибок

In [ ]:
MODEL_FOR_ERRORS = MODELS[0]
errs = pd.DataFrame(reports[MODEL_FOR_ERRORS]["errors"])
errs[["scenario", "error", "side", "item", "session"]] if len(errs) else "ошибок нет"

In [ ]:
failures = {f["scenario"]: f for f in reports[MODEL_FOR_ERRORS]["failures"]}
for sid in list(failures)[:5]:
    f = failures[sid]
    display(Markdown(f"**{sid}** · {f['category']} · {f['note']}"))
    for s in f["sessions"]:
        print("  диалог:", " | ".join(f"{t['role']}: {t['content']}" for t in s["turns_seen"]))
        print("  разделы:", s["gold_sections"], "→", s["classifier_sections"], "+ страховка", s["safety_net"])
        for c in s["calls"]:
            if c["task"].startswith("extract"):
                print(f"   [{c['task']}:{c['section']}] {c['raw'][:300]}")
        if s["expected"]:
            print("  ожидалось:", s["expected"]["facts"], s["expected"]["states"])
            print("  получено: ", s["facts"], s["states"], s["data_errors"])

## 8. E2E: пробы контекста и ответы

In [ ]:
rows = []
for m, r in reports.items():
    for p in r["probes"]:
        rows.append({"модель": m, "сценарий": p["scenario"], "запрос": p["text"],
                     "тип ок": sorted(p["gold_types"]) == sorted(p["pred_types"]),
                     "контекст (эталонный тип)": p["check_gold"]["ok"], "контекст (E2E)": p["check_llm"]["ok"],
                     "проверки ответа": " ".join("✓" if c["passed"] else "✗" for c in p["answer_checks"]) or "—"})
pd.DataFrame(rows)

In [ ]:
for p in reports[MODEL_FOR_ERRORS]["probes"]:
    if p["answer"]:
        display(Markdown(f"**{p['scenario']}** · {p['text']}  \n" +
                         "  \n".join(f"{'✓' if c['passed'] else '✗'} {c['why']}" for c in p["answer_checks"]) +
                         f"  \n> {p['answer'][:700]}"))

## 9. Отчёт

Все отчёты одним файлом — его можно прислать для разбора.

In [ ]:
bundle = {"commit": commit, "created": datetime.now().isoformat(timespec="seconds"), "reports": reports,
          "comparison": compare_reports(list(reports.values()))}
out = WORK / f"stage3_all_{datetime.now():%Y%m%d_%H%M}.json"
out.write_text(json.dumps(bundle, ensure_ascii=False, indent=1, default=list), encoding="utf-8")
print("сохранено:", out)
if IN_COLAB:
    from google.colab import files
    files.download(str(out))

> Демонстрационный проект. Данные синтетические, ответы ассистента не являются медицинской рекомендацией.